# 05 — RAG Retrieval: Similar Report Injection

**Idea:** At inference time, retrieve the top-k most similar training reports and inject them
as few-shot examples in the prompt. The model generates conditioned on:
1. The chest X-ray image
2. The clinical indication
3. k reference reports from semantically similar cases

**Retrieval strategy:** TF-IDF over clinical indications (CPU-only, no GPU required for indexing).  
Saves `reports/eval_hypotheses_rag_k{K}_{VARIANT}.json` and `reports/eval_metrics_rag_k{K}_{VARIANT}.json`
so notebook 04 STEP 7 can include RAG in the grand comparison.

**Run after notebook 04** (needs `eval_hypotheses_uniform_v3.json` cached references).

**Execution order across notebooks:**
```
04 (core eval)  →  [this notebook]  →  06 (assoc. rules)  →  04 STEP 7 (grand comparison)
```

## STEP 1 — Environment setup

In [ ]:
import os, sys, json, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import yaml

matplotlib.rcParams['figure.dpi'] = 120
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

PROCESSED_DIR  = REPO_ROOT / 'data' / 'processed'
CHECKPOINT_DIR = REPO_ROOT / 'checkpoints'
FIGURES_DIR    = REPO_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with open(REPO_ROOT / 'params.yaml') as f:
    params = yaml.safe_load(f)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU    : {props.name}  {props.total_memory/1e9:.1f} GB')

test_df = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
test_df = test_df[test_df['findings'].notna() & (test_df['findings'].str.strip() != '')].reset_index(drop=True)
references = test_df['findings'].str.strip().tolist()
print(f'Test set : {len(test_df)} studies')

train_df = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
train_df = train_df[train_df['findings'].notna() & (train_df['findings'].str.strip() != '')].reset_index(drop=True)
print(f'Train set: {len(train_df)} studies')

env_file = REPO_ROOT / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())
    import huggingface_hub
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        huggingface_hub.login(token=hf_token, add_to_git_credential=False)
        print('HF: logged in')

## STEP 2 — Build TF-IDF retrieval index

Indexes training set indications with TF-IDF bigrams (CPU-only).  
Retrieves by indication similarity — studies with similar clinical context tend to have similar findings.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train_indications = train_df['indication'].fillna('').astype(str).tolist()
train_findings    = train_df['findings'].str.strip().tolist()

tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
train_tfidf_matrix = tfidf.fit_transform(train_indications)
print(f'TF-IDF index: {train_tfidf_matrix.shape[0]} docs × {train_tfidf_matrix.shape[1]} features')

def retrieve_similar_reports(query_indication: str, k: int = 3) -> list[dict]:
    """Return the top-k training reports most similar to the query indication."""
    query_vec = tfidf.transform([query_indication])
    sims = cosine_similarity(query_vec, train_tfidf_matrix).flatten()
    top_k_idx = sims.argsort()[::-1][:k]
    return [
        {'indication': train_indications[i],
         'findings':   train_findings[i],
         'similarity': float(sims[i])}
        for i in top_k_idx
    ]

def build_rag_prompt(indication: str, retrieved: list[dict], k: int = 3) -> str:
    """Inject retrieved reports as few-shot context before the generation target."""
    parts = []
    if retrieved:
        parts.append('The following are findings from similar studies for reference:')
        for i, r in enumerate(retrieved[:k], 1):
            parts.append(f'  Example {i} (similarity={r["similarity"]:.2f}):')
            if r['indication'].strip():
                parts.append(f'    Indication: {r["indication"].strip()}')
            parts.append(f'    Findings: {r["findings"].strip()}')
        parts.append('')
    if indication.strip():
        parts.append(f'Indication: {indication.strip()}')
    parts.append('Findings:')
    return '\n'.join(parts)

# ── Smoke test ────────────────────────────────────────────────────────────────
TEST_QUERY = 'Patient with shortness of breath and low oxygen saturation, history of smoking'
retrieved = retrieve_similar_reports(TEST_QUERY, k=3)
print(f'\nQuery: "{TEST_QUERY}"\n')
for r in retrieved:
    print(f'  sim={r["similarity"]:.3f} | ind: {r["indication"][:80]}')
    print(f'             | fnd: {r["findings"][:90]}…')
print(f'\nPrompt preview:\n{build_rag_prompt(TEST_QUERY, retrieved)}')

## STEP 3 — RAG inference on test set

Loads the fine-tuned checkpoint, injects retrieved examples into each prompt,
and generates findings for all 600 test studies.  
Try `RAG_K = 3` (default) and optionally `RAG_K = 5` to measure diminishing returns.  
Results cached to `reports/eval_hypotheses_rag_k{K}_{VARIANT}.json`.

In [ ]:
from PIL import Image
from tqdm import tqdm
from peft import PeftModel
from src.training.model import load_model_and_processor

IMAGES_DIR = REPO_ROOT / params['data']['images_dir'] / 'images_normalized'
_BLANK = Image.new('RGB', (224, 224), color=(128, 128, 128))

SYSTEM_PROMPT = (
    'You are an expert radiologist. '
    'Write only the Findings section of a radiology report for the chest X-ray shown. '
    'Be concise and clinical. Do not include an Impression section.'
)

RAG_VARIANT = 'uniform_v3'
RAG_K_VALUES = [3]   # add 5 to compare: [3, 5]

def _load_image(row):
    frontal = row.get('frontal', [])
    if hasattr(frontal, '__len__') and len(frontal) > 0:
        p = IMAGES_DIR / list(frontal)[0]
        if p.exists():
            try: return Image.open(p).convert('RGB')
            except Exception: pass
    return _BLANK

rag_hypotheses = {}

for RAG_K in RAG_K_VALUES:
    variant_key = f'rag_k{RAG_K}_{RAG_VARIANT}'
    cache = REPO_ROOT / 'reports' / f'eval_hypotheses_{variant_key}.json'

    if cache.exists():
        rag_hypotheses[variant_key] = json.loads(cache.read_text())
        print(f'[{variant_key}] loaded {len(rag_hypotheses[variant_key])} cached hypotheses')
        continue

    print(f'\n[{variant_key}] loading model…')
    model, processor = load_model_and_processor(
        model_id=params['model']['base_model_id'],
        quantization=params['model']['quantization'],
    )
    adapter_path = CHECKPOINT_DIR / RAG_VARIANT / 'best_model'
    if not adapter_path.exists():
        adapter_path = CHECKPOINT_DIR / f'qlora_{RAG_VARIANT}' / 'best_model'
    model = PeftModel.from_pretrained(model, str(adapter_path))
    model.eval()

    hyps = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=variant_key):
        indication = str(row.get('indication', '') or '').strip()
        if indication.lower() in {'nan', 'none', ''}: indication = ''

        retrieved = retrieve_similar_reports(indication, k=RAG_K)
        user_text = build_rag_prompt(indication, retrieved, k=RAG_K)

        img = _load_image(row)
        content = [{'type': 'image'}, {'type': 'text', 'text': user_text}]
        prompt = processor.apply_chat_template(
            [{'role': 'user', 'content': content}],
            add_generation_prompt=True, tokenize=False,
        )
        inputs = processor(text=prompt, images=[img], return_tensors='pt', padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.inference_mode():
            out = model.generate(**inputs,
                max_new_tokens=params['model']['max_new_tokens'],
                do_sample=False, pad_token_id=processor.tokenizer.eos_token_id)
        hyp = processor.tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        hyps.append(hyp)

    cache.write_text(json.dumps(hyps, ensure_ascii=False, indent=2))
    rag_hypotheses[variant_key] = hyps
    del model, processor; torch.cuda.empty_cache()
    print(f'[{variant_key}] done — {len(hyps)} hypotheses saved')

print(f'\nRAG variants ready: {list(rag_hypotheses.keys())}')

## STEP 4 — Compute and save metrics

Writes `reports/eval_metrics_rag_k{K}_{VARIANT}.json` — the format notebook 04 STEP 7 auto-discovers.

In [ ]:
import bert_score.utils as _bsu
from bert_score import score as _bert_score
from src.data.labels import CHEXBERT_LABELS, run_chexbert
from sklearn.metrics import f1_score
import evaluate as hf_evaluate

_orig_sent_encode = _bsu.sent_encode
def _safe_sent_encode(tokenizer, sent):
    if getattr(tokenizer, 'model_max_length', 0) > 10_000:
        tokenizer.model_max_length = 512
    return _orig_sent_encode(tokenizer, sent)
_bsu.sent_encode = _safe_sent_encode

_bleu  = hf_evaluate.load('bleu')
_rouge = hf_evaluate.load('rouge')

def _compute_metrics(hyps, refs, variant):
    cache = REPO_ROOT / 'reports' / f'eval_metrics_{variant}.json'
    if cache.exists():
        print(f'  [{variant}] loaded from cache')
        return json.loads(cache.read_text())
    print(f'  [{variant}] BERTScore…')
    _, _, F = _bert_score(hyps, refs,
        model_type=params['eval']['bertscore_model'],
        lang='en', device=DEVICE, verbose=False, batch_size=16)
    print(f'  [{variant}] CheXbert…')
    hyp_labels = run_chexbert(hyps, uncertain_policy='present', device=DEVICE)
    ref_labels = run_chexbert(refs, uncertain_policy='present', device=DEVICE)
    print(f'  [{variant}] BLEU/ROUGE…')
    result = {
        'variant': variant,
        'bertscore_f1':      float(F.mean()),
        'chexbert_micro_f1': float(f1_score(ref_labels, hyp_labels, average='micro', zero_division=0)),
        'chexbert_macro_f1': float(f1_score(ref_labels, hyp_labels, average='macro', zero_division=0)),
        'bleu4':             _bleu.compute(predictions=hyps, references=[[r] for r in refs], max_order=4)['bleu'],
        'rouge_l':           _rouge.compute(predictions=hyps, references=refs)['rougeL'],
        'per_label_f1':      {label: float(f1_score(ref_labels[:, i], hyp_labels[:, i], zero_division=0))
                              for i, label in enumerate(CHEXBERT_LABELS)},
        'bertscore_per_study': F.tolist(),
    }
    cache.write_text(json.dumps(result, indent=2))
    print(f'  [{variant}] saved → {cache.name}')
    return result

rag_metrics = {}
for variant_key, hyps in rag_hypotheses.items():
    rag_metrics[variant_key] = _compute_metrics(hyps, references, variant_key)

_bsu.sent_encode = _orig_sent_encode
print('\nRAG metrics computed.')

## STEP 5 — Results and analysis

In [ ]:
# Load baseline for comparison
baseline_v3 = json.loads((REPO_ROOT / 'reports' / 'eval_metrics_uniform_v3.json').read_text())

DISPLAY_NAMES = {
    'uniform_v3':       'QLoRA uniform (v3) — no RAG',
    'rag_k3_uniform_v3': 'RAG k=3 (v3)',
    'rag_k5_uniform_v3': 'RAG k=5 (v3)',
}

comparison = {'uniform_v3': baseline_v3, **rag_metrics}
rows = []
for variant, m in comparison.items():
    rows.append({
        'Model':             DISPLAY_NAMES.get(variant, variant),
        'BERTScore-F1':      round(m['bertscore_f1'],      4),
        'CheXbert micro-F1': round(m['chexbert_micro_f1'], 4),
        'CheXbert macro-F1': round(m['chexbert_macro_f1'], 4),
        'BLEU-4':            round(m['bleu4'],             4),
        'ROUGE-L':           round(m['rouge_l'],           4),
    })
display(pd.DataFrame(rows).set_index('Model'))

# Delta vs no-RAG baseline
print('\nΔ BERTScore vs no-RAG baseline:')
for variant, m in rag_metrics.items():
    delta = m['bertscore_f1'] - baseline_v3['bertscore_f1']
    print(f'  {DISPLAY_NAMES.get(variant, variant)}: {delta:+.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
names  = [DISPLAY_NAMES.get(v, v) for v in comparison]
bscore = [comparison[v]['bertscore_f1'] for v in comparison]
colors = ['#1f77b4'] + ['#2ca02c'] * len(rag_metrics)
bars   = ax.bar(names, bscore, color=colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, bscore):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001, f'{val:.4f}',
            ha='center', va='bottom', fontsize=9)
ax.set_ylim(min(bscore) * 0.98, max(bscore) * 1.04)
ax.set_ylabel('BERTScore-F1'); ax.set_title('RAG Retrieval — impact on BERTScore-F1')
ax.tick_params(axis='x', labelrotation=15)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'eval_rag_comparison.png', dpi=150, bbox_inches='tight')
print('Saved eval_rag_comparison.png'); plt.show()

## Done

Metrics written to `reports/eval_metrics_rag_k*_uniform_v3.json`.  
**Next:** run notebook 06 (association rules), then return to notebook 04 STEP 7 for the grand comparison.